# SMS Spam Classification — NLP Preprocessing and Text Vectorization

This notebook uses the SMS Spam Collection dataset to practise five text-representation techniques:

1. One-Hot Encoding (token-level)
2. Bag of Words (manual implementation)
3. Count Vectorization (`CountVectorizer`)
4. N-Gram Vectorization
5. TF-IDF Vectorization

**Important:** The comments and commented-out POS/NER examples from the original notebook have been retained. Run the notebook from top to bottom. Keep one SMS as one document throughout the document-level vectorization sections.

## 1. Imports and NLTK resources

In [1]:
import pandas as pd
from pathlib import Path
from nltk.tokenize import word_tokenize, RegexpTokenizer
import sklearn as sk
from collections import Counter
import string
import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix

nltk.download("stopwords")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\goluh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\goluh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\goluh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\goluh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\goluh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     C:\Users\goluh\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_n

True

## 2. Load the SMS dataset

In [2]:
DATASET_PATH = Path("SMSSpamCollection")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATASET_PATH.resolve()}\n"
        "Place 'SMSSpamCollection' beside this notebook or update DATASET_PATH."
    )

df = pd.read_csv(
    DATASET_PATH,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="latin-1"
)

display(df.head())
print("Dataset shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Dataset shape: (5572, 2)
Label distribution:
 label
ham     4825
spam     747
Name: count, dtype: int64


## 3. Basic sentence and word splitting (original helper functions)

In [3]:
def sentence(st):
    return st.split('.')

def word(wo):
    return wo.split(' ')

## 4. Stemming experiment

In [4]:
# for st in df["message"]:
#     final=sentence(st)
#     final=[x for x in final if x!='']
#     print("Number of sentence ",len(final))

# for st in df["message"]:
#     stemmer=PorterStemmer()
#     stemmed_words = [stemmer.stem(st) for word in st]
#     print(stemmed_words)

# The following loop keeps the original Snowball stemming experiment,
# but stores the results and displays only a few messages to avoid huge output.
stemmed_docs = []
for st in df["message"]:
    tokenizer = RegexpTokenizer(r'\w+')
    cleaned_tokens = tokenizer.tokenize(st.lower())
    # print(cleaned_tokens)

    stemmer=SnowballStemmer('english')
    stemmed_words = [stemmer.stem(word) for word in cleaned_tokens if len(word)>1]
    stemmed_docs.append(stemmed_words)

# Show the first five processed messages.
for stemmed_words in stemmed_docs[:5]:
    print(stemmed_words)
    
    # stemmer=PorterStemmer()
    # stemmed_words = [stemmer.stem(word) for word in final]
    # print(stemmed_words)
    # print(len(final))
    # stemmed_words=[token for token in stemmed_words if token not in string.punctuation]
    # print("Number of words ",len(final))

['go', 'until', 'jurong', 'point', 'crazi', 'avail', 'onli', 'in', 'bugi', 'great', 'world', 'la', 'buffet', 'cine', 'there', 'got', 'amor', 'wat']
['ok', 'lar', 'joke', 'wif', 'oni']
['free', 'entri', 'in', 'wkli', 'comp', 'to', 'win', 'fa', 'cup', 'final', 'tkts', '21st', 'may', '2005', 'text', 'fa', 'to', '87121', 'to', 'receiv', 'entri', 'question', 'std', 'txt', 'rate', 'appli', '08452810075over18']
['dun', 'say', 'so', 'earli', 'hor', 'alreadi', 'then', 'say']
['nah', 'don', 'think', 'he', 'goe', 'to', 'usf', 'he', 'live', 'around', 'here', 'though']


## 5. Lemmatization and stopword removal

In [5]:
lemmatizer = WordNetLemmatizer()
tokenizer = RegexpTokenizer(r'\w+')
stop_word=set(stopwords.words('english'))

# Store the tokens for every SMS separately. Do not overwrite this list
# inside the loop, or only the final SMS will remain.
processed_docs = []

for st in df["message"]:
    cleaned_tokens = tokenizer.tokenize(st.lower())
    lemma_words = [lemmatizer.lemmatize(word) for word in cleaned_tokens if len(word)>1]
    removed_stopwords = [w for w in lemma_words if w not in stop_word]
    processed_docs.append(removed_stopwords)

    # pos_tags=[nltk.pos_tag(removed_stopwords)]
    # print(pos_tags[0][0][1])
    # print(pos_tags)
    # tag_count = Counter(tag for word, tag in pos_tags[0])

    # for tag, count in tag_count.items():
    #     print(f"{tag} appears {count} times")

    # print('//')
    # ner_tree = nltk.ne_chunk(pos_tags[0])
    # for subtree in ner_tree:
    #     if hasattr(subtree, "label"):
    #         entity = " ".join(word for word, tag in subtree.leaves())
    #         label = subtree.label()

    #         print(entity, ":", label)

print("Total processed SMS messages:", len(processed_docs))
print("First five processed messages:")
for doc in processed_docs[:5]:
    print(doc)

# Keep the original label column available for the later classification task.
y = df["label"].copy()

Total processed SMS messages: 5572
First five processed messages:
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'great', 'world', 'la', 'buffet', 'cine', 'got', 'amore', 'wat']
['ok', 'lar', 'joking', 'wif', 'oni']
['free', 'entry', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', '21st', 'may', '2005', 'text', 'fa', '87121', 'receive', 'entry', 'question', 'std', 'txt', 'rate', 'apply', '08452810075over18']
['dun', 'say', 'early', 'hor', 'already', 'say']
['nah', 'think', 'go', 'usf', 'life', 'around', 'though']


## 6. Prepare documents for vectorization

`processed_docs` is a list of token lists: each inner list belongs to one SMS. `CountVectorizer` and `TfidfVectorizer` expect a collection of document strings, so join each SMS's tokens back into one string. This does **not** combine all SMS messages into one document.

In [6]:
corpus = [" ".join(tokens) for tokens in processed_docs]

print("Number of documents:", len(corpus))
print("First five documents:")
for document in corpus[:5]:
    print(document)

Number of documents: 5572
First five documents:
go jurong point crazy available bugis great world la buffet cine got amore wat
ok lar joking wif oni
free entry wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry question std txt rate apply 08452810075over18
dun say early hor already say
nah think go usf life around though


## 7. One-Hot Encoding (OHE)

OHE creates one binary vector for each category/token. Here we fit the encoder on the unique vocabulary and demonstrate the encoded vector for a few tokens. The output is kept sparse to avoid allocating a very large dense matrix.

This is **token-level OHE**. A document containing several different words would have several active word positions (often called a multi-hot document representation), rather than a single 1 for the whole document.

In [7]:
# Collect all unique words from the processed SMS messages.
all_tokens = sorted({token for doc in processed_docs for token in doc})

# OneHotEncoder expects a 2D array: rows = samples, columns = categorical features.
unique_tokens = np.array(all_tokens, dtype=object).reshape(-1, 1)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
encoder.fit(unique_tokens)

# Encode a small sample so that the output stays readable.
sample_tokens = unique_tokens[:10]
encoded_sample = encoder.transform(sample_tokens)

print("Number of unique tokens:", len(all_tokens))
print("OHE matrix shape for sample:", encoded_sample.shape)
print("First few OHE feature names:", encoder.get_feature_names_out(["token"])[:20])
print("Sample encoded vectors:")
print(encoded_sample.toarray())

Number of unique tokens: 8089
OHE matrix shape for sample: (10, 8089)
First few OHE feature names: ['token_00' 'token_000' 'token_000pes' 'token_008704050406' 'token_0089'
 'token_0121' 'token_01223585236' 'token_01223585334' 'token_0125698789'
 'token_02' 'token_0207' 'token_02072069400' 'token_02073162414'
 'token_02085076972' 'token_021' 'token_03' 'token_04' 'token_0430'
 'token_05' 'token_050703']
Sample encoded vectors:
[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


## 8. Bag of Words (manual implementation)

Bag of Words represents each SMS using the frequency of each vocabulary word. Word order is ignored. Below, we build the count matrix manually with `Counter` and a sparse matrix.

In [8]:
# Build one shared vocabulary for all SMS messages.
vocabulary = sorted({word for doc in processed_docs for word in doc})
word_to_index = {word: index for index, word in enumerate(vocabulary)}

row_indices = []
col_indices = []
values = []

for row_index, tokens in enumerate(processed_docs):
    word_counts = Counter(tokens)
    for word, count in word_counts.items():
        row_indices.append(row_index)
        col_indices.append(word_to_index[word])
        values.append(count)

bow_matrix = csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(len(processed_docs), len(vocabulary)),
    dtype=np.int32
)

print("BoW matrix shape:", bow_matrix.shape)
print("First five SMS messages and first twenty vocabulary features:")
print(pd.DataFrame(
    bow_matrix[:5, :20].toarray(),
    columns=vocabulary[:20]
))

BoW matrix shape: (5572, 8089)
First five SMS messages and first twenty vocabulary features:
   00  000  000pes  008704050406  0089  0121  01223585236  01223585334  \
0   0    0       0             0     0     0            0            0   
1   0    0       0             0     0     0            0            0   
2   0    0       0             0     0     0            0            0   
3   0    0       0             0     0     0            0            0   
4   0    0       0             0     0     0            0            0   

   0125698789  02  0207  02072069400  02073162414  02085076972  021  03  04  \
0           0   0     0            0            0            0    0   0   0   
1           0   0     0            0            0            0    0   0   0   
2           0   0     0            0            0            0    0   0   0   
3           0   0     0            0            0            0    0   0   0   
4           0   0     0            0            0            0    0

## 9. Count Vectorization (`CountVectorizer`)

`CountVectorizer` automates the Bag-of-Words process: it learns a vocabulary and counts how often each vocabulary word appears in each document. Therefore, CountVectorizer is a standard implementation of a Bag-of-Words count representation, not a completely unrelated mathematical technique.

In [9]:
cv = CountVectorizer()

X_count = cv.fit_transform(corpus)

print("Vocabulary:", cv.get_feature_names_out()[:30])
print("Count matrix shape:", X_count.shape)
print("First five SMS messages and first twenty features:")
print(pd.DataFrame(
    X_count[:5, :20].toarray(),
    columns=cv.get_feature_names_out()[:20]
))

Vocabulary: ['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '0125698789' '02' '0207' '02072069400' '02073162414'
 '02085076972' '021' '03' '04' '0430' '05' '050703' '0578' '06' '07'
 '07008009200' '07046744435' '07090201529' '07090298926' '07099833605'
 '07123456789' '0721072']
Count matrix shape: (5572, 8082)
First five SMS messages and first twenty features:
   00  000  000pes  008704050406  0089  0121  01223585236  01223585334  \
0   0    0       0             0     0     0            0            0   
1   0    0       0             0     0     0            0            0   
2   0    0       0             0     0     0            0            0   
3   0    0       0             0     0     0            0            0   
4   0    0       0             0     0     0            0            0   

   0125698789  02  0207  02072069400  02073162414  02085076972  021  03  04  \
0           0   0     0            0            0            0    0   0   0   
1  

## 10. N-Gram Vectorization

An N-gram is a sequence of `n` consecutive tokens. Unigrams are individual words, bigrams are pairs of consecutive words, and trigrams are groups of three consecutive words.

- `(1, 1)`: unigrams only
- `(2, 2)`: bigrams only
- `(1, 2)`: unigrams and bigrams
- `(1, 3)`: unigrams, bigrams, and trigrams

In [10]:
ngram = CountVectorizer(ngram_range=(1, 2))

X_ngram = ngram.fit_transform(corpus)

print("N-gram features:", ngram.get_feature_names_out()[:40])
print("N-gram matrix shape:", X_ngram.shape)
print("First five SMS messages and first twenty features:")
print(pd.DataFrame(
    X_ngram[:5, :20].toarray(),
    columns=ngram.get_feature_names_out()[:20]
))

N-gram features: ['00' '00 easter' '00 per' '00 sub' '000' '000 bonus' '000 cash'
 '000 homeowner' '000 pound' '000 price' '000 prize' '000 xmas' '000pes'
 '000pes around' '008704050406' '008704050406 sp' '0089' '0089 last'
 '0121' '0121 2025050' '01223585236' '01223585236 xx' '01223585334'
 '01223585334 cum' '0125698789' '0125698789 ring' '02' '02 06' '02 09'
 '02 claimcode' '02 user' '0207' '0207 083' '0207 153' '02072069400'
 '02072069400 bx' '02073162414' '02073162414 cost' '02085076972'
 '02085076972 reply']
N-gram matrix shape: (5572, 39747)
First five SMS messages and first twenty features:
   00  00 easter  00 per  00 sub  000  000 bonus  000 cash  000 homeowner  \
0   0          0       0       0    0          0         0              0   
1   0          0       0       0    0          0         0              0   
2   0          0       0       0    0          0         0              0   
3   0          0       0       0    0          0         0              0   
4   0     

## 11. TF-IDF Vectorization

TF-IDF stands for **Term Frequency–Inverse Document Frequency**. It weights a term according to its frequency in one document and how common or rare it is across the full collection. The result is a weighted document-term matrix rather than raw counts.

In [11]:
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(corpus)

print("TF-IDF features:", tfidf.get_feature_names_out()[:30])
print("TF-IDF matrix shape:", X_tfidf.shape)
print("First five SMS messages and first twenty features:")
print(pd.DataFrame(
    X_tfidf[:5, :20].toarray(),
    columns=tfidf.get_feature_names_out()[:20]
).round(3))

TF-IDF features: ['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '0125698789' '02' '0207' '02072069400' '02073162414'
 '02085076972' '021' '03' '04' '0430' '05' '050703' '0578' '06' '07'
 '07008009200' '07046744435' '07090201529' '07090298926' '07099833605'
 '07123456789' '0721072']
TF-IDF matrix shape: (5572, 8082)
First five SMS messages and first twenty features:
    00  000  000pes  008704050406  0089  0121  01223585236  01223585334  \
0  0.0  0.0     0.0           0.0   0.0   0.0          0.0          0.0   
1  0.0  0.0     0.0           0.0   0.0   0.0          0.0          0.0   
2  0.0  0.0     0.0           0.0   0.0   0.0          0.0          0.0   
3  0.0  0.0     0.0           0.0   0.0   0.0          0.0          0.0   
4  0.0  0.0     0.0           0.0   0.0   0.0          0.0          0.0   

   0125698789   02  0207  02072069400  02073162414  02085076972  021   03  \
0         0.0  0.0   0.0          0.0          0.0          0.0  0.0  0.

## 12. Compare the five representations

| Technique | What is represented? | Main output |
|---|---|---|
| One-Hot Encoding | One individual token/category at a time | Binary token vector |
| Bag of Words | Word frequency in each document | Document-word count matrix |
| CountVectorizer | Automated Bag-of-Words implementation | Document-word count matrix |
| N-Grams | Counts of word sequences | Document n-gram matrix |
| TF-IDF | Weighted importance of words | Document-term weight matrix |

**Remember:** Bag of Words and CountVectorizer are closely related: the manual BoW cell builds the representation itself, while CountVectorizer performs the same kind of count-based vectorization for you.

## 13. Important: avoid data leakage when training a model

The vectorizer demonstrations above fit on the complete corpus so you can inspect their representations. For a real classification experiment, split the data first, fit the vectorizer only on the training messages, and use only `transform()` on the test messages. Do not use `fit_transform()` on the test set.

The example below prepares a TF-IDF representation safely; it does not train a classifier yet.

In [14]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"],
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Fit only on training messages, then transform the test messages.
tfidf_train_test = TfidfVectorizer()
X_train_tfidf = tfidf_train_test.fit_transform(X_train_text)
X_test_tfidf = tfidf_train_test.transform(X_test_text)

print(X_test_tfidf)
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14123 stored elements and shape (1115, 7699)>
  Coords	Values
  (0, 1599)	0.4298900463346133
  (0, 2536)	0.4793274570718033
  (0, 2963)	0.2486870967668338
  (0, 4274)	0.4681640202094909
  (0, 4416)	0.24443642010633784
  (0, 4724)	0.3537736303984626
  (0, 4789)	0.3003029448470412
  (0, 6885)	0.17126036120176288
  (1, 1285)	0.16799238866048902
  (1, 1595)	0.16113820516256716
  (1, 2034)	0.23468353589810237
  (1, 2689)	0.28247626082788124
  (1, 2887)	0.26432909369825824
  (1, 3640)	0.23757204912886298
  (1, 4241)	0.19746635613753163
  (1, 4416)	0.14023794524172808
  (1, 4817)	0.16307242557884438
  (1, 4909)	0.1808680086449985
  (1, 6352)	0.3177647904867511
  (1, 6587)	0.2431242017142982
  (1, 6764)	0.15334018050088336
  (1, 6769)	0.12102975275071497
  (1, 6853)	0.19452079201169858
  (1, 6911)	0.23263326501198553
  (1, 7326)	0.27675489661886965
  :	:
  (1111, 4199)	0.29394483784061326
  (1111, 5221)	0.3280648666144734
  (1111, 6

## 14. Notes for the next stage

1. Compare the five representations before selecting one for a model.
2. For SMS spam classification, keep the labels (`ham`/`spam`) aligned with their original messages.
3. Be cautious about removing stopwords: words such as `not` and `no` can carry meaning. You may want to retain them and compare model performance.
4. The current lemmatizer uses NLTK's default noun behaviour. POS-aware lemmatization can be explored separately.
5. For model evaluation, use a train/test split or cross-validation and fit all learned preprocessing/vectorization steps on training data only.